##### 加载数据集
- 数据预处理，需要拼接活动类型、标签节点到用户输入中；
- 构造平衡数据集（平衡数据集分布按照数量少的进行平衡性划分）：先取得更少label的数据集数量，再将其作为采样更多label的数据集的依据，最后将采样过后的数据集和更少的数据集合并即可；
- 创建训练集、验证集和测试集：根据7:1:2的比例构造数据集，注意先重置索引，因为需要根据比例计算两个数据集的结束位置，最后直接从上一个结束位置加载剩余数据集即可；
- 分批加载训练集、验证集、测试集；
- 数据预处理：对数据进行padding或truncation并将数据文本部分分别进行token化；

In [ ]:
import pandas as pd
from pandas.core.frame import DataFrame
import dashscope
from tqdm.notebook import tqdm
import random

In [ ]:
# 文本语义相似度去重

def semantic_dedup(dataset_df: DataFrame ,threshold: float) -> list[int]:
    """文本语义相似度去重

    Args:
        threshold (float): 相似度

    Returns:
        list[int]: 需要保留的数据索引
    """

    dataset_num = len(dataset_df)
    keep_indices = []
    removed_indices = set()
    
    pdqm = tqdm(range(1, dataset_num))

    for i in pdqm:
        if i not in removed_indices:
            keep_indices.append(i)
            
            for j in range(i+1, dataset_num, 500):
                response = dashscope.TextReRank.call(
                    model="qwen3-rerank",
                    query=dataset_df['Text'][i],
                    documents=dataset_df.iloc[j:j+500]['Text'].to_list(),
                    top_n=5,
                    return_documents=True,
                    api_key="sk-c94430ce01fe4c798b0a87399f079c0d",
                    instruct="Retrieve semantically similar text."
                )
                
                results = response['output']['results']
                for k in range(len(results)) :
                    if float(results[k]['relevance_score']) > threshold:
                        removed_indices.add(j+k)
                        pdqm.set_postfix(semantic_rate=(len(removed_indices) / dataset_num))
    return keep_indices

In [ ]:
# 重置标签
res_dataset = []

sc_reset_tag_a = ["(总)复选已报名","(总)复选未晋级","(总)复选未完赛"]
sc_reset_tag_b = ["大连-3月25周三19.00","大连-3月27周五19.00","大连-3月28周六10.30","大连-3月28周六14.30","大连-3月28周六19.00","大连-3月29周日10.30","大连-3月29周日14.30","大连-3月29周日19.00","沈阳-3月25周三19.00","沈阳-3月27周五19.00","沈阳-3月28周六10.30","沈阳-3月28周六14.30","沈阳-3月28周六19.00","营口-3月24周二19.00","沈阳-3月29周日10.30","沈阳-3月29周日14.30","沈阳-3月29周日19.00","营口-3月26周四19.00","营口-3月29周日16.00","营口-3月28周六9.00","营口-3月28周六16.00","营口-3月29周日9.00","锦州-3月24周二19.00","锦州-3月26周四19.00","锦州-3月28周六9.00","锦州-3月28周六16.00","锦州-3月29周日9.00","锦州-3月29周日16.00"]

sd_reset_tag_a = ["小学1-3年级(总)","小学4-6年级(总)","初中组(总)","高中组(总)"]
sd_reset_tag_b = ["活动已报名(总)","初选晋级(总)","初选未晋级(总)"]
sd_reset_tag_c = ["预选赛已报名(总)","预选赛未完赛(总)","预选赛末出席(总)","预选赛未晋级(总)"]
sd_reset_tag_d = ["预选赛补考资格(总)", "预选赛已购买(总)"]
sd_reset_tag_e = ["辽宁预选4月2周四19.00(总)","辽宁预选4月3周五19.00(总)"]

source_dataset_df = pd.read_csv(filepath_or_buffer="./trading_intent_classification_lora/lora_adapter/source_datasets_20260324.csv")
pdqm = tqdm(range(1, len(source_dataset_df)))
for i in pdqm:
    
    split_result = source_dataset_df['type'][i].split("、")
    for type in split_result:
        if type == '诗词':
            if source_dataset_df['tag_a'][i] == "1":
                tag_a_list = random.sample(sc_reset_tag_a, random.randint(1, len(sc_reset_tag_a))) + random.sample(sc_reset_tag_b, 1)
                # random.shuffle(tag_a_list)
                source_dataset_df['tag_a'][i] = "、".join(tag_a_list)
            res_dataset.append([
                f"Text:{source_dataset_df['text'][i]}\\nTag:{source_dataset_df['tag_a'][i]}",
                source_dataset_df['label'][i]
            ])
        elif type == '讲好中国故事':
            if source_dataset_df['tag_b'][i] == "1":
                tag_b_list = random.sample(sd_reset_tag_a, 1) + random.sample(sd_reset_tag_b, 1) + random.sample(sd_reset_tag_c, 1) + random.sample(sd_reset_tag_d, 1) + random.sample(sd_reset_tag_e, 1)
                # random.shuffle(tag_b_list)
                source_dataset_df['tag_b'][i] = "、".join(tag_b_list)
            res_dataset.append([
                f"Text:{source_dataset_df['text'][i]}\\nTag:{source_dataset_df['tag_b'][i]}",
                source_dataset_df['label'][i]
            ])
            
res_dataset_df = pd.DataFrame(data=res_dataset, columns=['Text', 'Label'])
res_dataset_df.to_csv(path_or_buf="./trading_intent_classification_lora/lora_adapter/datasets_20260324.csv", index=None)

In [ ]:
source_dataset_df = pd.read_csv(filepath_or_buffer="./trading_intent_classification_lora/lora_adapter/datasets_20260324.csv")
keep_indices = semantic_dedup(dataset_df=source_dataset_df, threshold=0.85)
source_dataset_df = source_dataset_df.iloc[keep_indices].reset_index(drop=True)
source_dataset_df.to_csv(path_or_buf="./trading_intent_classification_lora/lora_adapter/datasets_20260324.csv", index=None)

In [ ]:
# 检查数据集分布
dataset_df = pd.read_csv(filepath_or_buffer="./trading_intent_classification_lora/lora_adapter/datasets_20260324.csv", sep=",", header=None, names=["Text", "Label"])
dataset_df['Label'].value_counts()

In [ ]:
# 构造平衡数据集
def create_balance_df(df: DataFrame) -> DataFrame:
    """构造平衡数据集，先根据数量少的label确定另一种label的采样数量，再对双方采样相同数量数据集即可

    Args:
        df (DataFrame): 读取的CSV原始数据集

    Returns:
        DataFrame: 平衡数据集
    """
    num_has_no_intent = df[df['Label'] == "0"].shape[0] # 读取数量更少的label类型
    has_intent_subset = df[df['Label'] == "1"].sample(n=num_has_no_intent, random_state=123) # 根据数量更少的label类型随机采样另一种类型的数据集
    
    return pd.concat(objs=[has_intent_subset, df[df['Label'] == "0"]]) # 拼接两批数据即可
    
balance_dataste = create_balance_df(dataset_df)
balance_dataste['Label'] = balance_dataste['Label'].map({"1":1, "0":0}) # 数据类型转换
balance_dataste['Label'].value_counts()

In [ ]:
# 构造训练集、验证集和测试集
def dataset_random_split(df: DataFrame, train_frac: float, validation_frac: float) -> tuple[DataFrame, DataFrame, DataFrame]:
    """按照比例构造训练集、验证集和测试集（测试集比例自动推断）

    Args:
        df (DataFrame): 平衡数据集
        train_frac (float): 训练集比例
        validation_frac (float): 验证集比例

    Returns:
        tuple[DataFrame, DataFrame, DataFrame]: 训练集、验证集和测试集
    """
    df = df.sample(frac=1, random_state=312).reset_index(drop=True) # 重新设置索引
    train_sample_end_idx = int(len(df) * train_frac) # 设置训练集结束索引位置
    val_sample_end_idx = train_sample_end_idx + int(len(df) * validation_frac) # 设置验证集结束索引位置
    
    return df[:train_sample_end_idx], df[train_sample_end_idx: val_sample_end_idx], df[val_sample_end_idx:]

dataset_path = {
    "train": "./trading_intent_classification_lora/lora_adapter/train.csv",
    "val": "./trading_intent_classification_lora/lora_adapter/val.csv",
    "test": "./trading_intent_classification_lora/lora_adapter/test.csv"
}
# 训练集:验证集:测试集=7:1:2
train_dataset_df, val_dataset_df, test_dataset_df = dataset_random_split(df=balance_dataste, train_frac=0.7, validation_frac=0.2)
train_dataset_df.to_csv(path_or_buf=dataset_path['train'], index=None)
val_dataset_df.to_csv(path_or_buf=dataset_path['val'], index=None)
test_dataset_df.to_csv(path_or_buf=dataset_path['test'], index=None)